# Assistente de Reclamações com IA — classificação e avaliação

**Problema de negócio:** empresas de e-commerce e delivery recebem milhares de reclamações por dia. A equipe de atendimento precisa saber **quais tratar primeiro**, **quais temas mais geram insatisfação** e **como responder bem e rápido**.

**O que este notebook faz:**
1. Carrega avaliações reais de clientes (base pública Olist, e-commerce brasileiro) e separa as reclamações (notas 1 e 2 com comentário).
2. Usa um modelo de linguagem (API da OpenAI) para classificar cada reclamação por **tema, urgência, sentimento e risco de perder o cliente**, sugerir uma **ação** e uma **resposta**.
3. Salva o resultado em planilha para o painel de gestão (Etapa 3).

> A resposta sugerida é um **rascunho para um atendente revisar**, não um envio automático.

## 1. Configuração

**Chave da API:** no Colab, clique no ícone de chave (🔑 *Secrets*) na barra lateral, crie um segredo chamado `OPENAI_API_KEY` com a sua chave e ative o acesso para este notebook. **Nunca cole a chave direto no código** — o notebook vai para o GitHub.

**Dados:** baixe a base *Brazilian E-Commerce Public Dataset by Olist* no Kaggle e envie o arquivo `olist_order_reviews_dataset.csv` para o Colab (ícone de pasta 📁 na barra lateral).

In [1]:
!pip install -q openai pandas

In [2]:
import json, os, time
import pandas as pd
from openai import OpenAI

try:
    from google.colab import userdata          # rodando no Colab
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass                                       # rodando localmente: use a variável de ambiente

client = OpenAI()

MODELO = "gpt-4o-mini"   # modelo barato e rápido; troque se a sua conta usar outro
TAMANHO_AMOSTRA = 200    # comece pequeno para controlar o custo
SEMENTE = 42             # garante que a amostra seja sempre a mesma

## 2. Dados: das avaliações às reclamações

Usamos só avaliações com **nota 1 ou 2** e **comentário escrito**, porque é ali que estão as reclamações.

In [3]:
avaliacoes = pd.read_csv("olist_order_reviews_dataset.csv")
print(f"Avaliações na base: {len(avaliacoes):,}")

reclamacoes = (
    avaliacoes[avaliacoes["review_score"] <= 2]
    .dropna(subset=["review_comment_message"])
    .assign(texto=lambda d: (d["review_comment_title"].fillna("") + " " + d["review_comment_message"]).str.strip())
)
reclamacoes = reclamacoes[reclamacoes["texto"].str.len() >= 20]     # descarta textos curtos demais ("ruim")
reclamacoes = reclamacoes.drop_duplicates(subset="texto")
print(f"Reclamações com texto útil: {len(reclamacoes):,}")

amostra = reclamacoes.sample(n=min(TAMANHO_AMOSTRA, len(reclamacoes)), random_state=SEMENTE)
amostra = amostra[["review_id", "review_score", "review_creation_date", "texto"]].reset_index(drop=True)
amostra.head()

Avaliações na base: 99,224
Reclamações com texto útil: 10,188


,review_id,review_score,review_creation_date,texto
0,a3b1f50ee3bb0b052a187a0c1845bffd,1,2017-12-17 00:00:00,ainda estou esperando meu pedido.
1,63e7d80246143ac20128c947c898a3e8,1,2018-07-25 00:00:00,Insatisfação Estou muito insatisfeito com ess...
2,cc5b0a4875e99b76b467223755d062fd,1,2017-03-05 00:00:00,"o produto veio com defeito, aparência de usado!"
3,8ab6855b9fe9b812cd03a480a25058a1,2,2018-01-26 00:00:00,Foi entregue somente 1. Quero saber do outro p...
4,80e867a9c488688eb78b4f76d5e3c9f0,1,2018-03-06 00:00:00,o produto atrasou demais na entrega e além dis...


## 3. O prompt: instruções para a IA

Aqui está o coração do projeto (engenharia de prompt). As categorias e os critérios de urgência foram definidos com olhar de negócio, para que o resultado ajude a priorizar o atendimento.

In [4]:
TEMAS = [
    "Atraso na entrega",
    "Produto não entregue",
    "Produto errado ou incompleto",
    "Produto com defeito ou avariado",
    "Qualidade abaixo do esperado",
    "Cobrança, pagamento ou reembolso",
    "Atendimento ou falta de resposta",
    "Outros",
]

INSTRUCOES = f"""
Você é um analista de atendimento ao cliente de um e-commerce brasileiro.
Leia a reclamação do cliente e devolva APENAS um JSON com os campos abaixo.

- tema: um destes valores exatos: {TEMAS}
- tema_secundario: outro valor da mesma lista, ou null se não houver
- urgencia: "alta", "media" ou "baixa"
    alta  = produto não recebido, cobrança indevida, cliente cita reincidência,
            ameaça de Procon/processo ou pede cancelamento/reembolso
    media = atraso, produto errado ou com defeito ainda sem solução
    baixa = insatisfação com qualidade ou comentário sem pedido de solução
- sentimento: "muito negativo", "negativo" ou "neutro"
- risco_perder_cliente: "alto", "medio" ou "baixo"
- resumo: uma frase curta com o problema principal
- acao_sugerida: o próximo passo concreto para o time de atendimento
- resposta_sugerida: resposta educada e empática ao cliente, em português,
  com no máximo 70 palavras. Não prometa prazos ou valores específicos
  e não invente informações que não estão na reclamação.
"""

In [5]:
def classificar(texto: str, instrucoes: str = INSTRUCOES) -> dict:
    """Envia uma reclamação ao modelo e devolve a classificação como dicionário."""
    for tentativa in range(3):
        try:
            resp = client.chat.completions.create(
                model=MODELO,
                temperature=0,                                  # respostas consistentes
                response_format={"type": "json_object"},        # garante JSON válido
                messages=[
                    {"role": "system", "content": instrucoes},
                    {"role": "user", "content": texto},
                ],
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as erro:
            print(f"Tentativa {tentativa + 1} falhou: {erro}")
            time.sleep(2 ** tentativa)
    return {"tema": "ERRO"}

# Teste com uma reclamação antes de rodar a amostra inteira
print(amostra.loc[0, "texto"])
classificar(amostra.loc[0, "texto"])

ainda estou esperando meu pedido.


{'tema': 'Atraso na entrega',
 'tema_secundario': None,
 'urgencia': 'media',
 'sentimento': 'negativo',
 'risco_perder_cliente': 'medio',
 'resumo': 'Cliente ainda está esperando o pedido.',
 'acao_sugerida': 'Verificar o status da entrega e atualizar o cliente.',
 'resposta_sugerida': 'Olá! Lamentamos pelo atraso na entrega do seu pedido. Estamos verificando a situação e faremos o possível para resolver isso rapidamente. Agradecemos pela sua paciência.'}

## 4. Classificar a amostra

Com 200 reclamações e o modelo `gpt-4o-mini`, o custo costuma ficar em centavos de dólar. Confira o consumo no painel da OpenAI.

In [6]:
def classificar_lote(textos, instrucoes=INSTRUCOES):
    resultados = []
    for i, texto in enumerate(textos, start=1):
        resultados.append(classificar(texto, instrucoes))
        if i % 20 == 0:
            print(f"{i} de {len(textos)} classificadas")
    return pd.DataFrame(resultados)

# Se você já tem o resultado da 1ª rodada, envie o CSV para o Colab: ele é reaproveitado, sem custo
if os.path.exists("reclamacoes_classificadas.csv"):
    classificado = pd.read_csv("reclamacoes_classificadas.csv", encoding="utf-8-sig")
    print("Resultado anterior carregado do arquivo.")
else:
    classificado = pd.concat([amostra, classificar_lote(amostra["texto"])], axis=1)
    classificado.to_csv("reclamacoes_classificadas.csv", index=False, encoding="utf-8-sig")
classificado.head()

Resultado anterior carregado do arquivo.


,review_id,review_score,review_creation_date,texto,tema,tema_secundario,urgencia,sentimento,risco_perder_cliente,resumo,acao_sugerida,resposta_sugerida
0,a3b1f50ee3bb0b052a187a0c1845bffd,1,2017-12-17 00:00:00,ainda estou esperando meu pedido.,Atraso na entrega,NaN,media,negativo,medio,Cliente ainda está esperando o pedido.,Verificar o status da entrega e atualizar o cl...,Olá! Lamentamos pelo atraso na entrega do seu ...
1,63e7d80246143ac20128c947c898a3e8,1,2018-07-25 00:00:00,Insatisfação Estou muito insatisfeito com ess...,Atendimento ou falta de resposta,NaN,baixa,negativo,alto,Cliente insatisfeito com a falta de resposta d...,Investigar a situação do atendimento ao client...,Lamentamos saber da sua insatisfação e da falt...
2,cc5b0a4875e99b76b467223755d062fd,1,2017-03-05 00:00:00,"o produto veio com defeito, aparência de usado!",Produto com defeito ou avariado,NaN,media,negativo,medio,Produto recebido com defeito e aparência de us...,Solicitar a troca ou devolução do produto.,Olá! Lamentamos muito que o produto tenha cheg...
3,8ab6855b9fe9b812cd03a480a25058a1,2,2018-01-26 00:00:00,Foi entregue somente 1. Quero saber do outro p...,Produto não entregue,NaN,media,negativo,medio,Cliente recebeu apenas um produto e questiona ...,Verificar o status do pedido e informar o clie...,Olá! Agradecemos seu contato. Vamos verificar ...
4,80e867a9c488688eb78b4f76d5e3c9f0,1,2018-03-06 00:00:00,o produto atrasou demais na entrega e além dis...,Atraso na entrega,Produto errado ou incompleto,media,negativo,medio,Atraso na entrega e falta de um cartucho de ti...,"Investigar o atraso e a entrega incompleta, of...",Olá! Lamentamos pelo atraso na entrega e pela ...


## 5. Primeira leitura dos resultados

In [7]:
print("Reclamações por tema:")
print(classificado["tema"].value_counts(), "\n")

print("Reclamações por urgência:")
print(classificado["urgencia"].value_counts(), "\n")

print("Urgência x risco de perder o cliente:")
pd.crosstab(classificado["urgencia"], classificado["risco_perder_cliente"])

Reclamações por tema:
tema
Produto não entregue                55
Atraso na entrega                   47
Produto errado ou incompleto        46
Qualidade abaixo do esperado        23
Produto com defeito ou avariado     21
Cobrança, pagamento ou reembolso     4
Atendimento ou falta de resposta     2
Outros                               2
Name: count, dtype: int64 

Reclamações por urgência:
urgencia
media    93
alta     78
baixa    29
Name: count, dtype: int64 

Urgência x risco de perder o cliente:


risco_perder_cliente,alto,baixo,medio
urgencia,,,
alta,77,0,1
baixa,3,22,4
media,1,0,92


**Leitura da versão 1:** a IA classificou as 200 sem erro, mas o cruzamento acima mostra que o **risco de perder o cliente** está quase copiando a **urgência** (alta→alto, média→médio). Os dois campos deveriam medir coisas diferentes. Também há confusão entre *atraso* e *não entregue* e entre *não entregue* e *incompleto*. As próximas seções medem esses erros e corrigem o prompt.

## 6. Etapa 2 — gabarito humano

Para saber se a IA acerta, comparamos com uma classificação feita por uma pessoa (o **gabarito**).

1. Rode a célula abaixo e baixe o arquivo `gabarito_para_preencher.csv` (ícone de pasta 📁 → três pontinhos → *Fazer download*).
2. Abra no Google Planilhas ou no Excel e preencha **sem olhar a resposta da IA**:
   - `tema_humano`: o **número** do tema (1 a 8, lista impressa abaixo)
   - `urgencia_humana`: `alta`, `media` ou `baixa`
3. Salve como CSV com o nome `gabarito_preenchido.csv` e envie de volta para o Colab.

In [8]:
N_GABARITO = 30
gabarito = classificado.loc[:N_GABARITO - 1, ["review_id", "texto"]].copy()   # a amostra já é aleatória
gabarito["tema_humano"] = ""
gabarito["urgencia_humana"] = ""
gabarito.to_csv("gabarito_para_preencher.csv", index=False, sep=";", encoding="utf-8-sig")

print("Temas:")
for n, t in enumerate(TEMAS, start=1):
    print(f"  {n} = {t}")
print("Urgência: alta, media ou baixa")

Temas:
  1 = Atraso na entrega
  2 = Produto não entregue
  3 = Produto errado ou incompleto
  4 = Produto com defeito ou avariado
  5 = Qualidade abaixo do esperado
  6 = Cobrança, pagamento ou reembolso
  7 = Atendimento ou falta de resposta
  8 = Outros
Urgência: alta, media ou baixa


## 7. Quanto a IA concorda com o gabarito (versão 1)

In [9]:
import unicodedata

def normalizar(valor):
    """Tira acentos, espaços e maiúsculas: 'Média ' vira 'media'."""
    texto = unicodedata.normalize("NFKD", str(valor)).encode("ascii", "ignore").decode()
    return texto.strip().lower()

def ler_gabarito(arquivo: str) -> pd.DataFrame:
    """Lê o gabarito preenchido e avisa se algum valor foi digitado errado."""
    g = pd.read_csv(arquivo, sep=None, engine="python", encoding="utf-8-sig", dtype=str)
    g = g.dropna(subset=["tema_humano", "urgencia_humana"]).reset_index(drop=True)   # ignora linhas em branco
    g["tema_humano"] = g["tema_humano"].str.strip().astype(int).map(lambda n: TEMAS[n - 1])
    g["urgencia_humana"] = g["urgencia_humana"].map(normalizar)
    invalidos = g[~g["urgencia_humana"].isin(["alta", "media", "baixa"])]
    if len(invalidos):
        print("⚠️ Urgência digitada errado (corrija no CSV e rode de novo):")
        print(invalidos[["review_id", "urgencia_humana"]], "\n")
    return g

def concordancia(gabarito: pd.DataFrame, previsto: pd.DataFrame, rotulo: str):
    """Compara a classificação da IA com o gabarito e mostra onde ela errou."""
    base = gabarito[["review_id", "texto", "tema_humano", "urgencia_humana"]].merge(
        previsto[["review_id", "tema", "urgencia"]], on="review_id")
    base["urgencia"] = base["urgencia"].map(normalizar)
    acerto_tema = (base["tema"] == base["tema_humano"]).mean()
    acerto_urg = (base["urgencia"] == base["urgencia_humana"]).mean()
    print(f"{rotulo}: tema {acerto_tema:.0%} · urgência {acerto_urg:.0%} (n = {len(base)})")
    erros = base[(base["tema"] != base["tema_humano"]) | (base["urgencia"] != base["urgencia_humana"])]
    return {"tema": acerto_tema, "urgencia": acerto_urg}, erros

preenchido = ler_gabarito("gabarito_preenchido.csv")
metricas_v1, erros_v1 = concordancia(preenchido, classificado, "Versão 1")
pd.set_option("display.max_colwidth", 120)
erros_v1

Versão 1: tema 60% · urgência 50% (n = 30)


,review_id,texto,tema_humano,urgencia_humana,tema,urgencia
0,a3b1f50ee3bb0b052a187a0c1845bffd,ainda estou esperando meu pedido.,Produto não entregue,alta,Atraso na entrega,media
1,63e7d80246143ac20128c947c898a3e8,"Insatisfação Estou muito insatisfeito com essa loja, qualificação zero, não compro mais nesta loja se fosse o stark...",Atendimento ou falta de resposta,alta,Atendimento ou falta de resposta,baixa
4,80e867a9c488688eb78b4f76d5e3c9f0,"o produto atrasou demais na entrega e além disso a entrega não foi completa, pois faltou um cartucho de tinta colori...",Produto errado ou incompleto,alta,Atraso na entrega,media
5,5b7b8a7aa529492160c8bf209291ed8d,Postagem A residência fica uma pessoa 24 horas. Um absurdo redirecionarem para os Correios. Alegando que tentaram 3 ...,Outros,baixa,Atraso na entrega,media
6,56db2bc65ad713e2b4732f39bf1b4ccb,"Prazo de Entrga muito longo , e quando chega a data prevista , não entrega. - PessimoParceiro",Produto não entregue,alta,Atraso na entrega,media
7,cd49e5e8f4cedc28fa9b0ab941150f60,O produto veio com a caixa do Lego amassada e é presente. Não é porque não pedi embrulho que não tenham que ter o de...,Produto com defeito ou avariado,media,Produto com defeito ou avariado,baixa
8,7c1d09034917d55f41618ce69ced8c50,Fiquei dececionada o produto É bem diferente na foto é lindos ao vivo é bem fajuto eu não compro mas esses produtos ...,Qualidade abaixo do esperado,alta,Qualidade abaixo do esperado,baixa
9,7071ca4c66b85b5182ab667c2f2b9739,Eu gostei muito dos meus produto,Outros,baixa,Qualidade abaixo do esperado,baixa
10,94e4bde9a16194c468881a8f4bb08195,não confere o produto Eu comprei o KIT cartuchos 664XL (preto e Colorido) e mandaram ERRADO Kit de cartuchos 662XL ....,Produto errado ou incompleto,alta,Produto errado ou incompleto,media
11,b13013b811c2325934e8e049683bdcf2,Paguei mais caro e esperei mais tempo pra receber escovas Dourada e Branca...\n\nNo fim das contas recebi 2 escovas ...,Atraso na entrega,baixa,Produto errado ou incompleto,media


## 8. Versão 2 do prompt

Ajustes feitos a partir dos erros da versão 1:
- **definição de cada tema**, separando *não entregue* (o cliente ainda não recebeu) de *atraso* (recebeu, mas fora do prazo) e de *incompleto* (faltou parte do pedido);
- regra para quando há **dois problemas**: o tema principal é o que ainda está sem solução;
- **risco de perder o cliente com critérios próprios**, independentes da urgência.

In [10]:
INSTRUCOES_V2 = f"""
Você é um analista de atendimento ao cliente de um e-commerce brasileiro.
Leia a reclamação do cliente e devolva APENAS um JSON com os campos abaixo.

- tema: um destes valores exatos: {TEMAS}
    Atraso na entrega = o produto CHEGOU, mas depois do prazo prometido
    Produto não entregue = o cliente diz que AINDA NÃO RECEBEU o pedido
        (inclui "ainda estou esperando", "não chegou", "consta entregue mas não recebi")
    Produto errado ou incompleto = chegou item diferente do comprado ou faltou
        parte do pedido (ex.: comprou 2 e recebeu 1)
    Produto com defeito ou avariado = chegou quebrado, danificado ou não funciona
    Qualidade abaixo do esperado = funciona, mas é pior do que o anunciado
    Cobrança, pagamento ou reembolso = valor, estorno, nota fiscal ou devolução do dinheiro
    Atendimento ou falta de resposta = a queixa principal é a loja não responder ou atender mal
    Outros = nenhum dos anteriores
  Se houver mais de um problema, o tema é o que ainda está SEM SOLUÇÃO
  e o outro vai em tema_secundario.
- tema_secundario: outro valor da mesma lista, ou null se não houver
- urgencia: "alta", "media" ou "baixa"
    alta  = produto não recebido, cobrança indevida, cliente cita reincidência,
            ameaça de Procon/processo ou pede cancelamento/reembolso
    media = atraso, produto errado, incompleto ou com defeito ainda sem solução
    baixa = insatisfação com qualidade ou comentário sem pedido de solução
- sentimento: "muito negativo", "negativo" ou "neutro"
- risco_perder_cliente: avalie SEPARADAMENTE da urgência, pelo que o cliente demonstra
    alto  = diz que não compra mais, pede cancelamento, ameaça Procon/processo,
            cita reincidência ou usa tom de revolta
    medio = está insatisfeito, mas ainda espera uma solução da loja
    baixo = crítica pontual, em tom moderado
  Um problema urgente pode ter risco médio se o cliente só quer a solução.
- resumo: uma frase curta com o problema principal
- acao_sugerida: o próximo passo concreto para o time de atendimento
- resposta_sugerida: resposta educada e empática ao cliente, em português,
  com no máximo 70 palavras. Não prometa prazos ou valores específicos
  e não invente informações que não estão na reclamação.
"""

In [11]:
v2_gabarito = pd.concat([preenchido[["review_id"]],
                         classificar_lote(preenchido["texto"], INSTRUCOES_V2)], axis=1)
metricas_v2, erros_v2 = concordancia(preenchido, v2_gabarito, "Versão 2")
erros_v2

20 de 30 classificadas
Versão 2: tema 70% · urgência 63% (n = 30)


,review_id,texto,tema_humano,urgencia_humana,tema,urgencia
3,8ab6855b9fe9b812cd03a480a25058a1,Foi entregue somente 1. Quero saber do outro produto.,Produto não entregue,media,Produto errado ou incompleto,media
4,80e867a9c488688eb78b4f76d5e3c9f0,"o produto atrasou demais na entrega e além disso a entrega não foi completa, pois faltou um cartucho de tinta colori...",Produto errado ou incompleto,alta,Atraso na entrega,media
5,5b7b8a7aa529492160c8bf209291ed8d,Postagem A residência fica uma pessoa 24 horas. Um absurdo redirecionarem para os Correios. Alegando que tentaram 3 ...,Outros,baixa,Atraso na entrega,media
6,56db2bc65ad713e2b4732f39bf1b4ccb,"Prazo de Entrga muito longo , e quando chega a data prevista , não entrega. - PessimoParceiro",Produto não entregue,alta,Atraso na entrega,media
8,7c1d09034917d55f41618ce69ced8c50,Fiquei dececionada o produto É bem diferente na foto é lindos ao vivo é bem fajuto eu não compro mas esses produtos ...,Qualidade abaixo do esperado,alta,Qualidade abaixo do esperado,baixa
9,7071ca4c66b85b5182ab667c2f2b9739,Eu gostei muito dos meus produto,Outros,baixa,Qualidade abaixo do esperado,baixa
10,94e4bde9a16194c468881a8f4bb08195,não confere o produto Eu comprei o KIT cartuchos 664XL (preto e Colorido) e mandaram ERRADO Kit de cartuchos 662XL ....,Produto errado ou incompleto,alta,Produto errado ou incompleto,media
11,b13013b811c2325934e8e049683bdcf2,Paguei mais caro e esperei mais tempo pra receber escovas Dourada e Branca...\n\nNo fim das contas recebi 2 escovas ...,Atraso na entrega,baixa,Produto errado ou incompleto,media
17,8c45e97be6c3aa74ac2b6f49aba8a869,Recomendo não comprar produtos que são entregue por outras empresas que não forem das lojas lannister.,Outros,baixa,Qualidade abaixo do esperado,baixa
18,ca25630895145a0b552eadc98a5ab445,"luminária faltando eu comprei 2luminarias, so que vocês só me mandaram 1 luminária, mandei um email pra vocês faland...",Produto errado ou incompleto,alta,Produto errado ou incompleto,media


## 9. Versão 3 do prompt

Os erros que sobraram na versão 2 eram, na maioria, de **urgência**, e não de leitura do texto. A regra de urgência do prompt dizia que produto errado ou incompleto é *média*, mas, na visão de gestão do gabarito, **o cliente que pagou e ficou sem o que comprou é prioridade alta**. A versão 3 alinha a regra a esse critério de negócio e trata três casos que a IA confundia:
- comentário **elogioso ou sem problema** (nota baixa por engano) → *Outros*, urgência baixa;
- **ter que retirar nos Correios** ou problema de frete sem atraso → *Outros*;
- crítica genérica à loja, sem problema concreto → *Outros*.

In [12]:
INSTRUCOES_V3 = f"""
Você é um analista de atendimento ao cliente de um e-commerce brasileiro.
Leia a reclamação do cliente e devolva APENAS um JSON com os campos abaixo.

- tema: um destes valores exatos: {TEMAS}
    Atraso na entrega = o produto CHEGOU, mas depois do prazo prometido
    Produto não entregue = o cliente diz que AINDA NÃO RECEBEU o pedido
        (inclui "ainda estou esperando", "não chegou", "passou o prazo e nada",
        "consta entregue mas não recebi")
    Produto errado ou incompleto = chegou item diferente do comprado ou faltou
        parte do pedido (ex.: comprou 2 e recebeu 1)
    Produto com defeito ou avariado = chegou quebrado, danificado ou não funciona
    Qualidade abaixo do esperado = funciona, mas é pior do que o anunciado
    Cobrança, pagamento ou reembolso = valor, estorno, nota fiscal ou devolução do dinheiro
    Atendimento ou falta de resposta = a queixa principal é a loja não responder ou atender mal
    Outros = nenhum dos anteriores, incluindo: comentário elogioso ou sem problema,
        ter que retirar o produto nos Correios, problema de frete sem atraso
        e crítica genérica à loja sem um problema concreto
  Se houver mais de um problema, o tema é o que ainda está SEM SOLUÇÃO
  e o outro vai em tema_secundario.
- tema_secundario: outro valor da mesma lista, ou null se não houver
- urgencia: "alta", "media" ou "baixa"
    alta  = o cliente pagou e está SEM o que comprou (não recebeu, recebeu
            o item errado ou faltou parte do pedido), cobrança indevida,
            diz que não compra mais, cita reincidência ou ameaça Procon/processo
    media = produto chegou com defeito ou avariado, ou outro problema
            que ainda precisa de solução da loja
    baixa = o problema já se resolveu (ex.: atrasou, mas chegou), insatisfação
            com qualidade sem pedido de solução, elogio ou crítica genérica
- sentimento: "muito negativo", "negativo" ou "neutro"
- risco_perder_cliente: avalie SEPARADAMENTE da urgência, pelo que o cliente demonstra
    alto  = diz que não compra mais, pede cancelamento, ameaça Procon/processo,
            cita reincidência ou usa tom de revolta
    medio = está insatisfeito, mas ainda espera uma solução da loja
    baixo = crítica pontual, em tom moderado
  Um problema urgente pode ter risco médio se o cliente só quer a solução.
- resumo: uma frase curta com o problema principal
- acao_sugerida: o próximo passo concreto para o time de atendimento
- resposta_sugerida: resposta educada e empática ao cliente, em português,
  com no máximo 70 palavras. Não prometa prazos ou valores específicos
  e não invente informações que não estão na reclamação.
"""

In [13]:
v3_gabarito = pd.concat([preenchido[["review_id"]],
                         classificar_lote(preenchido["texto"], INSTRUCOES_V3)], axis=1)
metricas_v3, erros_v3 = concordancia(preenchido, v3_gabarito, "Versão 3")
erros_v3

20 de 30 classificadas
Versão 3: tema 77% · urgência 73% (n = 30)


,review_id,texto,tema_humano,urgencia_humana,tema,urgencia
3,8ab6855b9fe9b812cd03a480a25058a1,Foi entregue somente 1. Quero saber do outro produto.,Produto não entregue,media,Produto errado ou incompleto,media
4,80e867a9c488688eb78b4f76d5e3c9f0,"o produto atrasou demais na entrega e além disso a entrega não foi completa, pois faltou um cartucho de tinta colori...",Produto errado ou incompleto,alta,Atraso na entrega,media
5,5b7b8a7aa529492160c8bf209291ed8d,Postagem A residência fica uma pessoa 24 horas. Um absurdo redirecionarem para os Correios. Alegando que tentaram 3 ...,Outros,baixa,Produto não entregue,alta
6,56db2bc65ad713e2b4732f39bf1b4ccb,"Prazo de Entrga muito longo , e quando chega a data prevista , não entrega. - PessimoParceiro",Produto não entregue,alta,Atraso na entrega,media
8,7c1d09034917d55f41618ce69ced8c50,Fiquei dececionada o produto É bem diferente na foto é lindos ao vivo é bem fajuto eu não compro mas esses produtos ...,Qualidade abaixo do esperado,alta,Qualidade abaixo do esperado,baixa
11,b13013b811c2325934e8e049683bdcf2,Paguei mais caro e esperei mais tempo pra receber escovas Dourada e Branca...\n\nNo fim das contas recebi 2 escovas ...,Atraso na entrega,baixa,Produto errado ou incompleto,alta
14,4dd68a0dee5ede8a445f797c46bca436,o acabento que estava na foto era bonito a que chegou foi de um sintetico feio com um acabento todo borrado orrivel ...,Qualidade abaixo do esperado,baixa,Qualidade abaixo do esperado,media
16,214deaabf69eb3d26c7bae540e03d1b2,Demora demasiada na entrega,Atraso na entrega,media,Atraso na entrega,baixa
21,303a4aa26fbc5324b7d910caf7441741,"Comprei a mochila em janeiro, acompanhei status de transporte até 31/01/2018, com previsão de entrega para 14/02/201...",Outros,media,Produto não entregue,alta
25,6fc621ab5e65648a02d62f6b7ae6054e,"Todas as etapas de entrega tiveram um problema, o objeto chegou com atraso em tudo e até agora consta que não chegou...",Atraso na entrega,alta,Produto não entregue,alta


## 10. Validação em reclamações novas

As versões 2 e 3 foram ajustadas **olhando** as 30 reclamações do gabarito, então o acerto nelas tende a ser otimista. Para medir de forma justa, usamos **20 reclamações que nenhuma versão do prompt viu**.

1. Rode a célula abaixo e baixe `validacao_para_preencher.csv`.
2. Preencha do mesmo jeito (número do tema e urgência), **sem olhar a IA**.
3. Envie de volta como `validacao_preenchida.csv` e rode as células seguintes.

In [14]:
validacao = classificado.loc[N_GABARITO:N_GABARITO + 19, ["review_id", "texto"]].copy()
validacao["tema_humano"] = ""
validacao["urgencia_humana"] = ""
validacao.to_csv("validacao_para_preencher.csv", index=False, sep=";", encoding="utf-8-sig")
print(f"{len(validacao)} reclamações para preencher.")

20 reclamações para preencher.


In [15]:
validacao_preenchida = ler_gabarito("validacao_preenchido.csv")
val_v3 = pd.concat([validacao_preenchida[["review_id"]].reset_index(drop=True),
                    classificar_lote(validacao_preenchida["texto"], INSTRUCOES_V3)], axis=1)

print("Gabarito (30 reclamações usadas para ajustar o prompt):")
for nome, m in [("Versão 1", metricas_v1), ("Versão 2", metricas_v2), ("Versão 3", metricas_v3)]:
    print(f"  {nome}: tema {m['tema']:.0%} · urgência {m['urgencia']:.0%}")
print("\nValidação (20 reclamações novas):")
val_m1, _ = concordancia(validacao_preenchida, classificado, "  Versão 1")
val_m3, erros_val = concordancia(validacao_preenchida, val_v3, "  Versão 3")
erros_val

20 de 20 classificadas
Gabarito (30 reclamações usadas para ajustar o prompt):
  Versão 1: tema 60% · urgência 50%
  Versão 2: tema 70% · urgência 63%
  Versão 3: tema 77% · urgência 73%

Validação (20 reclamações novas):
  Versão 1: tema 30% · urgência 60% (n = 20)
  Versão 3: tema 60% · urgência 75% (n = 20)


,review_id,texto,tema_humano,urgencia_humana,tema,urgencia
2,b577000cc81238c033e48d29029b3089,"Bom produto, só não coube o notebook de 15,6'. Vou ter que devolver.",Outros,baixa,Produto errado ou incompleto,baixa
3,32c4a269079211ca93dc6bfd0dff84a5,cancelamento recebi o produto no dia 09 de maio diferente do que tinha pedido e fiz o pedido de cancelamento no dia ...,Atendimento ou falta de resposta,alta,Produto errado ou incompleto,alta
4,7acfea10e5045832e525b1b1b8262063,"Entrega incorreta Boa tarde, recebi apenas 1 par de luvas x11, como posso entrar em contato para receber o segundo p...",Atendimento ou falta de resposta,media,Produto errado ou incompleto,media
5,ad1cbb9246426cdb97e55878ccb16dc4,O produto tinha a aparência de metal não tinha informações e era boracha,Outros,baixa,Qualidade abaixo do esperado,baixa
8,247e5873bde75f075031ff46770972d3,O pedido foi entregue no prazo\r\nMais o pedido tem o acabamento e a qualidade ruim e a mala pequena veio com defeito,Produto com defeito ou avariado,alta,Produto com defeito ou avariado,media
12,4ed6a83a1c64b54be8e93267e1995004,"Eu tive que retirar no correio, sendo que paguei o frete. É a segunda encomenda que tenho que retirar nos correios, ...",Outros,baixa,Outros,alta
13,554fc78b087cf9e06cb15f1631c3d63b,Loja possui o produto mas não deseja enviar.,Outros,media,Outros,baixa
14,65e5e0396daf89b147152d2e1b23b958,"Apesar do produto ter chegado no prazo, as medidas não se adequaram ao que foi informado no site em ""Dados Técnicos""...",Outros,baixa,Qualidade abaixo do esperado,baixa
15,c7fb23cd8188e9ed9136d6b84ec117c6,estou aguardando o produto chegar,Produto não entregue,baixa,Produto não entregue,alta
16,36494c1e2c7202fdfe61bc28283dee0e,"A entrega não foi concluída e não tive uma posição da loja ainda, até o momento aguardo respostas do fornecedor.",Atendimento ou falta de resposta,alta,Produto não entregue,alta


## 11. Classificação final

Com a melhor versão, reclassificamos a amostra inteira. Esse arquivo alimenta o painel da Etapa 3.

In [16]:
classificado_final = pd.concat([amostra, classificar_lote(amostra["texto"], INSTRUCOES_V3)], axis=1)
classificado_final.to_csv("reclamacoes_classificadas_final.csv", index=False, encoding="utf-8-sig")

print(classificado_final["tema"].value_counts(), "\n")
print(classificado_final["urgencia"].value_counts(), "\n")
print("Urgência x risco (versão final):")
pd.crosstab(classificado_final["urgencia"], classificado_final["risco_perder_cliente"])

20 de 200 classificadas
40 de 200 classificadas
60 de 200 classificadas
80 de 200 classificadas
100 de 200 classificadas
120 de 200 classificadas
140 de 200 classificadas
160 de 200 classificadas
180 de 200 classificadas
200 de 200 classificadas
tema
Produto não entregue                76
Produto errado ou incompleto        51
Produto com defeito ou avariado     21
Qualidade abaixo do esperado        20
Atraso na entrega                   14
Outros                              11
Atendimento ou falta de resposta     5
Cobrança, pagamento ou reembolso     2
Name: count, dtype: int64 

urgencia
alta     122
media     43
baixa     35
Name: count, dtype: int64 

Urgência x risco (versão final):


risco_perder_cliente,alto,baixo,medio
urgencia,,,
alta,98,0,24
baixa,2,33,0
media,1,0,42


## Próximas etapas

- **Etapa 3 — painel:** gráficos com os indicadores (temas, urgência, risco) e exemplos de respostas sugeridas.
- **Etapa 4 — publicação:** README com problema, método, resultados (v1 → v3, gabarito e validação) e limitações; prints e GitHub.